# PokéAtlas — Mejora de retrieval v2

Esta versión corrige dos problemas detectados durante las pruebas:

1. los chunks grandes de movimientos/habilidades perdían precisión semántica;
2. una ficha por entidad exige 1.247 embeddings y supera el límite diario del nivel gratuito de Gemini.

La solución es usar **micro-chunks de 2 entidades de la misma categoría**.

Esto reduce movimientos + habilidades a aproximadamente **624 embeddings**, manteniendo los nombres oficiales en español e inglés dentro de cada fragmento.

> Importante: antes de modificar ChromaDB se realiza una prueba mínima de cuota. Si la cuota diaria de Gemini todavía no se ha renovado, el notebook se detiene sin borrar nada.


In [1]:
from pathlib import Path
import os
import time
import re
import pandas as pd
import requests
from io import StringIO

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma


## 1. Abrir el proyecto y la base Chroma existente

In [2]:
CWD = Path.cwd()

if (CWD / "data").exists() and (CWD / ".env").exists():
    ROOT = CWD
elif (CWD.parent / "data").exists() and (CWD.parent / ".env").exists():
    ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "No encuentro la raíz de PokeAtlas. Comprueba data/ y .env."
    )

DATA_DIR = ROOT / "data" / "knowledge"
CHROMA_DIR = ROOT / "chroma_db"

load_dotenv(ROOT / ".env")

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("No se ha encontrado GEMINI_API_KEY en .env")

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vectorstore = Chroma(
    collection_name="pokeatlas_knowledge",
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

print("✅ ChromaDB abierta.")
print("Vectores actuales:", vectorstore._collection.count())


✅ ChromaDB abierta.
Vectores actuales: 471


## 2. Comprobar cuota ANTES de modificar Chroma

Esta prueba consume una única solicitud.

Si aparece un error `429` indicando `requests per day limit exceeded`, **no continúes**. La base existente seguirá intacta.


In [3]:
try:
    probe = embeddings.embed_query("Prueba de disponibilidad de Gemini Embeddings.")
    print("✅ La cuota de Gemini Embeddings está disponible.")
    print("Dimensiones:", len(probe))
except Exception as exc:
    message = str(exc)

    if "per day limit exceeded" in message.lower():
        raise RuntimeError(
            "La cuota diaria de Gemini Embeddings todavía no se ha renovado. "
            "Detén aquí el notebook y vuelve a probar cuando Google permita "
            "nuevas solicitudes. ChromaDB NO ha sido modificada."
        ) from exc

    raise


✅ La cuota de Gemini Embeddings está disponible.
Dimensiones: 3072


## 3. Cargar movimientos, habilidades y nombres oficiales en español

In [4]:
moves_df = pd.read_csv(DATA_DIR / "moves.csv")
abilities_df = pd.read_csv(DATA_DIR / "abilities.csv")

POKEAPI_CSV_BASE = (
    "https://raw.githubusercontent.com/"
    "PokeAPI/pokeapi/master/data/v2/csv"
)

def load_remote_csv(filename, timeout=30):
    url = f"{POKEAPI_CSV_BASE}/{filename}"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return pd.read_csv(StringIO(response.text))

move_names = load_remote_csv("move_names.csv")
ability_names = load_remote_csv("ability_names.csv")

SPANISH_LANGUAGE_ID = 7

move_names_es = (
    move_names[move_names["local_language_id"] == SPANISH_LANGUAGE_ID]
    [["move_id", "name"]]
    .rename(columns={"name": "name_es"})
    .drop_duplicates("move_id")
)

ability_names_es = (
    ability_names[ability_names["local_language_id"] == SPANISH_LANGUAGE_ID]
    [["ability_id", "name"]]
    .rename(columns={"name": "name_es"})
    .drop_duplicates("ability_id")
)

moves_es = moves_df.merge(
    move_names_es, left_on="id", right_on="move_id", how="left"
)

abilities_es = abilities_df.merge(
    ability_names_es, left_on="id", right_on="ability_id", how="left"
)

print("Movimientos:", len(moves_es))
print("Habilidades:", len(abilities_es))
print("Nombres españoles de movimientos:", moves_es["name_es"].notna().sum())
print("Nombres españoles de habilidades:", abilities_es["name_es"].notna().sum())


Movimientos: 937
Habilidades: 310
Nombres españoles de movimientos: 937
Nombres españoles de habilidades: 310


## 4. Crear fichas compactas individuales

In [5]:
TYPE_ES = {
    "normal":"normal","fire":"fuego","water":"agua","electric":"eléctrico",
    "grass":"planta","ice":"hielo","fighting":"lucha","poison":"veneno",
    "ground":"tierra","flying":"volador","psychic":"psíquico","bug":"bicho",
    "rock":"roca","ghost":"fantasma","dragon":"dragón","dark":"siniestro",
    "steel":"acero","fairy":"hada",
}

DAMAGE_CLASS_ES = {
    "physical":"físico","special":"especial","status":"estado"
}

def safe(value, default="no consta"):
    if pd.isna(value):
        return default
    value = str(value).strip()
    return value if value else default

def pretty(value):
    return safe(value).replace("-", " ")

move_docs = []

for _, row in moves_es.iterrows():
    en = safe(row.get("move"))
    es = safe(row.get("name_es"), pretty(en))
    type_en = safe(row.get("type"))
    damage_en = safe(row.get("damage_class"))

    text = (
        f"Movimiento Pokémon: {es}. "
        f"Nombre inglés: {pretty(en)}. Identificador: {en}. "
        f"Generación: {safe(row.get('generation'))}. "
        f"Tipo: {TYPE_ES.get(type_en, type_en)} ({type_en}). "
        f"Categoría: {DAMAGE_CLASS_ES.get(damage_en, damage_en)} ({damage_en}). "
        f"Potencia: {safe(row.get('power'))}. "
        f"Precisión: {safe(row.get('accuracy'))}. "
        f"PP: {safe(row.get('pp'))}. "
        f"Prioridad: {safe(row.get('priority'))}. "
        f"Efecto: {safe(row.get('short_effect'))}."
    )

    move_docs.append(
        Document(
            page_content=text,
            metadata={
                "source":"moves.csv",
                "category":"move",
                "name":en,
                "name_es":es,
            }
        )
    )

ability_docs = []

for _, row in abilities_es.iterrows():
    en = safe(row.get("ability"))
    es = safe(row.get("name_es"), pretty(en))

    text = (
        f"Habilidad Pokémon: {es}. "
        f"Nombre inglés: {pretty(en)}. Identificador: {en}. "
        f"Generación: {safe(row.get('generation'))}. "
        f"Serie principal: {safe(row.get('is_main_series'))}. "
        f"Efecto: {safe(row.get('short_effect'))}."
    )

    ability_docs.append(
        Document(
            page_content=text,
            metadata={
                "source":"abilities.csv",
                "category":"ability",
                "name":en,
                "name_es":es,
            }
        )
    )

print("Fichas de movimientos:", len(move_docs))
print("Fichas de habilidades:", len(ability_docs))


Fichas de movimientos: 937
Fichas de habilidades: 310


## 5. Agrupar de 2 en 2 sin mezclar categorías

Un micro-chunk contiene como máximo **dos movimientos** o **dos habilidades**.

Así reducimos el consumo diario sin volver a los chunks grandes que perjudicaban la recuperación.


In [6]:
def pair_documents(docs):
    paired = []

    for i in range(0, len(docs), 2):
        group = docs[i:i+2]

        content = "\n\n---\n\n".join(doc.page_content for doc in group)

        paired.append(
            Document(
                page_content=content,
                metadata={
                    "source": group[0].metadata["source"],
                    "category": group[0].metadata["category"],
                    "names": " | ".join(doc.metadata["name"] for doc in group),
                    "names_es": " | ".join(doc.metadata["name_es"] for doc in group),
                    "records": len(group),
                    "entity_granularity": "pair",
                }
            )
        )

    return paired

move_pairs = pair_documents(move_docs)
ability_pairs = pair_documents(ability_docs)

docs_to_add = move_pairs + ability_pairs

print("Micro-chunks de movimientos:", len(move_pairs))
print("Micro-chunks de habilidades:", len(ability_pairs))
print("Total de embeddings necesarios:", len(docs_to_add))

assert len(docs_to_add) < 1000


Micro-chunks de movimientos: 469
Micro-chunks de habilidades: 155
Total de embeddings necesarios: 624


## 6. Sustituir únicamente movimientos y habilidades

Esta celda se ejecuta **solo después** de haber confirmado en el punto 2 que Gemini acepta nuevas solicitudes.


In [7]:
before = vectorstore._collection.count()

vectorstore._collection.delete(where={"category": "move"})
vectorstore._collection.delete(where={"category": "ability"})

after = vectorstore._collection.count()

print("Vectores antes:", before)
print("Vectores base conservados (Pokémon + tipos):", after)
print("Vectores move/ability eliminados:", before - after)


Vectores antes: 471
Vectores base conservados (Pokémon + tipos): 231
Vectores move/ability eliminados: 240


## 7. Indexar los micro-chunks

Usamos tandas moderadas y pausas para respetar también el límite de tokens/minuto.

Si se detecta específicamente el **límite diario**, el notebook se detiene en lugar de esperar inútilmente durante varios minutos.


In [8]:
BATCH_SIZE = 40
PAUSE_SECONDS = 65
MAX_RETRIES = 5

def retry_delay_from_error(message, default=65):
    match = re.search(
        r"retry in\s+([0-9.]+)s",
        message,
        flags=re.IGNORECASE
    )
    if match:
        return max(float(match.group(1)) + 2, default)
    return default

total = len(docs_to_add)

for start in range(0, total, BATCH_SIZE):
    end = min(start + BATCH_SIZE, total)
    batch = docs_to_add[start:end]

    print(f"\nIndexando micro-chunks {start + 1}-{end} de {total}...")

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            vectorstore.add_documents(batch)

            print(
                f"✅ Tanda completada. "
                f"Vectores totales: {vectorstore._collection.count()}"
            )
            break

        except Exception as exc:
            message = str(exc)
            lower = message.lower()

            if "per day limit exceeded" in lower:
                raise RuntimeError(
                    "Se ha alcanzado la cuota DIARIA de Gemini Embeddings. "
                    "No tiene sentido reintentar ahora. Los lotes ya guardados "
                    "permanecen en ChromaDB."
                ) from exc

            if "429" in message or "RESOURCE_EXHAUSTED" in message:
                if attempt == MAX_RETRIES:
                    raise

                pause = retry_delay_from_error(message, PAUSE_SECONDS)

                print(
                    f"⚠️ Cuota temporal alcanzada "
                    f"(intento {attempt}/{MAX_RETRIES})."
                )
                print(f"Pausa de {pause:.0f} s y reintento...")
                time.sleep(pause)
            else:
                raise

    if end < total:
        print("Pausa automática entre tandas...")
        time.sleep(PAUSE_SECONDS)

print("\n✅ Indexación completada.")
print("Vectores finales:", vectorstore._collection.count())



Indexando micro-chunks 1-40 de 624...
✅ Tanda completada. Vectores totales: 271
Pausa automática entre tandas...

Indexando micro-chunks 41-80 de 624...
✅ Tanda completada. Vectores totales: 311
Pausa automática entre tandas...

Indexando micro-chunks 81-120 de 624...
✅ Tanda completada. Vectores totales: 351
Pausa automática entre tandas...

Indexando micro-chunks 121-160 de 624...
✅ Tanda completada. Vectores totales: 391
Pausa automática entre tandas...

Indexando micro-chunks 161-200 de 624...
✅ Tanda completada. Vectores totales: 431
Pausa automática entre tandas...

Indexando micro-chunks 201-240 de 624...
✅ Tanda completada. Vectores totales: 471
Pausa automática entre tandas...

Indexando micro-chunks 241-280 de 624...
✅ Tanda completada. Vectores totales: 511
Pausa automática entre tandas...

Indexando micro-chunks 281-320 de 624...
✅ Tanda completada. Vectores totales: 551
Pausa automática entre tandas...

Indexando micro-chunks 321-360 de 624...
✅ Tanda completada. Vectores

## 8. Comprobar retrieval

In [9]:
def buscar(query, k=4):
    resultados = vectorstore.similarity_search(query, k=k)

    print(f"Pregunta: {query}")
    print("=" * 80)

    for i, doc in enumerate(resultados, 1):
        label = (
            doc.metadata.get("names_es")
            or doc.metadata.get("names")
            or doc.metadata.get("name")
            or "sin nombre"
        )

        print(
            f"\nRESULTADO {i} | "
            f"{doc.metadata.get('category')} | {label}"
        )
        print("-" * 80)
        print(doc.page_content[:1400])

    return resultados


In [10]:
buscar(
    "¿Qué potencia tiene el movimiento Terremoto y de qué tipo es?",
    k=4
)


Pregunta: ¿Qué potencia tiene el movimiento Terremoto y de qué tipo es?

RESULTADO 1 | move | Terremoto | Fisura
--------------------------------------------------------------------------------
Movimiento Pokémon: Terremoto. Nombre inglés: earthquake. Identificador: earthquake. Generación: gen-i. Tipo: tierra (ground). Categoría: físico (physical). Potencia: 100.0. Precisión: 100.0. PP: 10.0. Prioridad: 0. Efecto: Inflicts regular damage and can hit Dig users..

---

Movimiento Pokémon: Fisura. Nombre inglés: fissure. Identificador: fissure. Generación: gen-i. Tipo: tierra (ground). Categoría: físico (physical). Potencia: no consta. Precisión: 30.0. PP: 5.0. Prioridad: 0. Efecto: Causes a one-hit KO..

RESULTADO 2 | move | Mil Temblores | Fuerza Telúrica
--------------------------------------------------------------------------------
Movimiento Pokémon: Mil Temblores. Nombre inglés: thousand waves. Identificador: thousand-waves. Generación: gen-vi. Tipo: tierra (ground). Categoría: fís

[Document(id='104c2143-924a-4a3a-a00f-490b4ed72fdc', metadata={'records': 2, 'names': 'earthquake | fissure', 'names_es': 'Terremoto | Fisura', 'entity_granularity': 'pair', 'source': 'moves.csv', 'category': 'move'}, page_content='Movimiento Pokémon: Terremoto. Nombre inglés: earthquake. Identificador: earthquake. Generación: gen-i. Tipo: tierra (ground). Categoría: físico (physical). Potencia: 100.0. Precisión: 100.0. PP: 10.0. Prioridad: 0. Efecto: Inflicts regular damage and can hit Dig users..\n\n---\n\nMovimiento Pokémon: Fisura. Nombre inglés: fissure. Identificador: fissure. Generación: gen-i. Tipo: tierra (ground). Categoría: físico (physical). Potencia: no consta. Precisión: 30.0. PP: 5.0. Prioridad: 0. Efecto: Causes a one-hit KO..'),
 Document(id='122394b4-6f86-439d-aa62-b7fcda527aac', metadata={'names_es': 'Mil Temblores | Fuerza Telúrica', 'category': 'move', 'entity_granularity': 'pair', 'records': 2, 'names': 'thousand-waves | lands-wrath', 'source': 'moves.csv'}, page_

In [11]:
buscar(
    "¿Qué efecto tiene la habilidad Rough Skin?",
    k=4
)


Pregunta: ¿Qué efecto tiene la habilidad Rough Skin?

RESULTADO 1 | ability | Sombra Trampa | Piel Tosca
--------------------------------------------------------------------------------
Habilidad Pokémon: Sombra Trampa. Nombre inglés: shadow tag. Identificador: shadow-tag. Generación: gen-iii. Serie principal: 1. Efecto: Prevents opponents from fleeing or switching out..

---

Habilidad Pokémon: Piel Tosca. Nombre inglés: rough skin. Identificador: rough-skin. Generación: gen-iii. Serie principal: 1. Efecto: Damages attacking Pokémon for 1/8 their max HP on contact..

RESULTADO 2 | ability | Piel Seca | Descarga
--------------------------------------------------------------------------------
Habilidad Pokémon: Piel Seca. Nombre inglés: dry skin. Identificador: dry-skin. Generación: gen-iv. Serie principal: 1. Efecto: Causes 1/8 max HP in damage each turn during strong sunlight, but heals for 1/8 max HP during rain.  Increases damage from Fire moves to 1.25×, but absorbs Water moves, he

[Document(id='bdb1360c-f64a-4c9b-863b-bbd77a011d54', metadata={'category': 'ability', 'records': 2, 'source': 'abilities.csv', 'names_es': 'Sombra Trampa | Piel Tosca', 'names': 'shadow-tag | rough-skin', 'entity_granularity': 'pair'}, page_content='Habilidad Pokémon: Sombra Trampa. Nombre inglés: shadow tag. Identificador: shadow-tag. Generación: gen-iii. Serie principal: 1. Efecto: Prevents opponents from fleeing or switching out..\n\n---\n\nHabilidad Pokémon: Piel Tosca. Nombre inglés: rough skin. Identificador: rough-skin. Generación: gen-iii. Serie principal: 1. Efecto: Damages attacking Pokémon for 1/8 their max HP on contact..'),
 Document(id='a4621fe2-ddc3-40fc-9b9a-1edd92e24daf', metadata={'names_es': 'Piel Seca | Descarga', 'names': 'dry-skin | download', 'entity_granularity': 'pair', 'category': 'ability', 'records': 2, 'source': 'abilities.csv'}, page_content="Habilidad Pokémon: Piel Seca. Nombre inglés: dry skin. Identificador: dry-skin. Generación: gen-iv. Serie principal

In [12]:
buscar(
    "¿Qué efecto tiene la habilidad Piel Tosca?",
    k=4
)


Pregunta: ¿Qué efecto tiene la habilidad Piel Tosca?

RESULTADO 1 | ability | Sombra Trampa | Piel Tosca
--------------------------------------------------------------------------------
Habilidad Pokémon: Sombra Trampa. Nombre inglés: shadow tag. Identificador: shadow-tag. Generación: gen-iii. Serie principal: 1. Efecto: Prevents opponents from fleeing or switching out..

---

Habilidad Pokémon: Piel Tosca. Nombre inglés: rough skin. Identificador: rough-skin. Generación: gen-iii. Serie principal: 1. Efecto: Damages attacking Pokémon for 1/8 their max HP on contact..

RESULTADO 2 | ability | Piel Seca | Descarga
--------------------------------------------------------------------------------
Habilidad Pokémon: Piel Seca. Nombre inglés: dry skin. Identificador: dry-skin. Generación: gen-iv. Serie principal: 1. Efecto: Causes 1/8 max HP in damage each turn during strong sunlight, but heals for 1/8 max HP during rain.  Increases damage from Fire moves to 1.25×, but absorbs Water moves, he

[Document(id='bdb1360c-f64a-4c9b-863b-bbd77a011d54', metadata={'names_es': 'Sombra Trampa | Piel Tosca', 'category': 'ability', 'names': 'shadow-tag | rough-skin', 'records': 2, 'source': 'abilities.csv', 'entity_granularity': 'pair'}, page_content='Habilidad Pokémon: Sombra Trampa. Nombre inglés: shadow tag. Identificador: shadow-tag. Generación: gen-iii. Serie principal: 1. Efecto: Prevents opponents from fleeing or switching out..\n\n---\n\nHabilidad Pokémon: Piel Tosca. Nombre inglés: rough skin. Identificador: rough-skin. Generación: gen-iii. Serie principal: 1. Efecto: Damages attacking Pokémon for 1/8 their max HP on contact..'),
 Document(id='a4621fe2-ddc3-40fc-9b9a-1edd92e24daf', metadata={'category': 'ability', 'records': 2, 'names': 'dry-skin | download', 'source': 'abilities.csv', 'entity_granularity': 'pair', 'names_es': 'Piel Seca | Descarga'}, page_content="Habilidad Pokémon: Piel Seca. Nombre inglés: dry skin. Identificador: dry-skin. Generación: gen-iv. Serie principal

## Resultado esperado

El objetivo es que los primeros resultados incluyan claramente:

- **Terremoto / earthquake**
- **Piel Tosca / rough-skin**

Si esto ocurre, cerramos la fase de recuperación y pasamos al RAG con Gemini + LangGraph + memoria.
